# Fase 5 — Avaliação
Os cinco critérios de sucesso da Fase 1, cobrados um por vez sobre o K-Means com k = 5.

In [ ]:
import joblib
import numpy as np
import pandas as pd
from pathlib import Path
from scipy.optimize import linear_sum_assignment
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score, silhouette_samples

SEED = 42
RAIZ = Path('..')
PROCESSADOS = RAIZ / 'dados' / 'processados'
ESTADO = RAIZ / 'estado'
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 250)

X_modelo = pd.read_parquet(PROCESSADOS / 'X_modelo.parquet')
base = pd.read_parquet(PROCESSADOS / 'base_lojas_clusters.parquet')
modelo_k5 = joblib.load(RAIZ / 'modelos' / 'kmeans_k5.joblib')
FEATURES_MODELO = X_modelo.columns.tolist()

## 1. Estabilidade

In [2]:
referencia = base['cluster']
K = 5

# rodar de novo com a mesma semente e a mesma base muda algo?
repeticoes_iguais = [adjusted_rand_score(referencia, KMeans(n_clusters=K, n_init=20, random_state=SEED).fit_predict(X_modelo)) for _ in range(5)]
print('ARI em 5 reexecuções com a mesma semente e a mesma base:', repeticoes_iguais)

ARI em 5 reexecuções com a mesma semente e a mesma base: [1.0, 1.0, 1.0, 1.0, 1.0]


In [3]:
def alinhar(rotulos_ref, rotulos_novos, k=K):
    # renomeia os clusters novos para os números da referência (o K-Means numera em ordem arbitrária)
    cruzado = pd.crosstab(rotulos_ref, rotulos_novos).reindex(index=range(k), columns=range(k), fill_value=0)
    linhas, colunas = linear_sum_assignment(-cruzado.values)
    return pd.Series(rotulos_novos).map(dict(zip(colunas, linhas))).values

N_REP = 30
ari = {'semente': [], 'reordenacao': [], 'subamostra80': []}
trocas = {'semente': [], 'reordenacao': [], 'subamostra80': []}
permanencia = pd.DataFrame(0, index=X_modelo.index, columns=['presente', 'mesmo_cluster'])
inercias_semente = []
for i in range(N_REP):
    m = KMeans(n_clusters=K, n_init=20, random_state=1000 + i).fit(X_modelo)
    inercias_semente.append(m.inertia_)
    ari['semente'].append(adjusted_rand_score(referencia, m.labels_))
    trocas['semente'].append((alinhar(referencia.values, m.labels_) != referencia.values).mean())

    emb = X_modelo.sample(frac=1, random_state=2000 + i)
    rot = pd.Series(KMeans(n_clusters=K, n_init=20, random_state=SEED).fit_predict(emb), index=emb.index).loc[referencia.index]
    ari['reordenacao'].append(adjusted_rand_score(referencia, rot))
    trocas['reordenacao'].append((alinhar(referencia.values, rot.values) != referencia.values).mean())

for i in range(100):
    sub = X_modelo.sample(frac=0.8, random_state=3000 + i)
    rot = KMeans(n_clusters=K, n_init=20, random_state=SEED).fit_predict(sub)
    ref_sub = referencia.loc[sub.index]
    alinhado = alinhar(ref_sub.values, rot)
    if i < N_REP:
        ari['subamostra80'].append(adjusted_rand_score(ref_sub, rot))
        trocas['subamostra80'].append((alinhado != ref_sub.values).mean())
    permanencia.loc[sub.index, 'presente'] += 1
    permanencia.loc[sub.index, 'mesmo_cluster'] += (alinhado == ref_sub.values).astype(int)

resumo_estab = pd.DataFrame({
    'ari_medio': {t: np.mean(v) for t, v in ari.items()},
    'ari_minimo': {t: np.min(v) for t, v in ari.items()},
    'execucoes_com_ari_abaixo_0.80': {t: int((np.array(v) < 0.80).sum()) for t, v in ari.items()},
    'pct_lojas_que_trocam_medio': {t: np.mean(v) * 100 for t, v in trocas.items()},
    'pct_lojas_que_trocam_maximo': {t: np.max(v) * 100 for t, v in trocas.items()},
})
print(resumo_estab.round(3).to_string())

              ari_medio  ari_minimo  execucoes_com_ari_abaixo_0.80  pct_lojas_que_trocam_medio  pct_lojas_que_trocam_maximo
semente           0.900       0.742                              3                       5.142                       13.812
reordenacao       0.907       0.706                              2                       4.771                       16.502
subamostra80      0.754       0.502                             16                      14.159                       30.157


In [4]:
# as execuções instáveis são ruído ou uma segunda solução concorrente?
print('ARI por semente (ordenado):', np.sort(np.round(ari['semente'], 3)))
print('inércias distintas encontradas:', sorted(set(np.round(inercias_semente, 1))))
print('ARI por subamostra (ordenado):', np.sort(np.round(ari['subamostra80'], 3)))

ARI por semente (ordenado): [0.742 0.767 0.799 0.805 0.805 0.896 0.902 0.903 0.917 0.918 0.919 0.921
 0.923 0.927 0.929 0.929 0.929 0.929 0.929 0.929 0.929 0.929 0.929 0.929
 0.929 0.929 0.929 0.93  0.93  0.93 ]
inércias distintas encontradas: [np.float64(3884.1), np.float64(3884.2), np.float64(3884.3), np.float64(3884.4), np.float64(3884.6), np.float64(3886.9), np.float64(3887.1), np.float64(3889.5), np.float64(3892.7)]
ARI por subamostra (ordenado): [0.502 0.511 0.524 0.529 0.53  0.535 0.549 0.603 0.638 0.672 0.691 0.717
 0.743 0.766 0.768 0.794 0.801 0.84  0.86  0.891 0.892 0.9   0.901 0.902
 0.905 0.908 0.917 0.925 0.952 0.966]


In [5]:
# estabilidade por loja: em quantas das subamostras em que apareceu ela ficou no mesmo cluster
permanencia['taxa'] = permanencia['mesmo_cluster'] / permanencia['presente']
permanencia['silhueta'] = silhouette_samples(X_modelo, referencia)
permanencia['cluster'] = referencia
faixas = pd.cut(permanencia['taxa'], [-0.01, 0.5, 0.8, 0.95, 1.0], labels=['até 50%', '50-80%', '80-95%', 'acima de 95%'])
print('lojas por taxa de permanência (100 subamostras de 80%):')
print(faixas.value_counts().sort_index().to_frame('lojas').assign(pct=lambda d: (d['lojas'] / len(permanencia) * 100).round(1)).to_string())
por_cluster = permanencia.groupby('cluster').agg(lojas=('taxa', 'size'), permanencia_media=('taxa', 'mean'),
                                                 pct_lojas_abaixo_80=('taxa', lambda s: (s < 0.8).mean() * 100),
                                                 silhueta_media=('silhueta', 'mean'))
print('\npor cluster:'); print(por_cluster.round(3).to_string())
print('\ncorrelação de Spearman entre silhueta da loja e taxa de permanência:',
      round(permanencia['silhueta'].corr(permanencia['taxa'], method='spearman'), 3))
print('silhueta média - lojas estáveis (>95%):', round(permanencia.loc[permanencia['taxa'] > 0.95, 'silhueta'].mean(), 3),
      '| instáveis (<80%):', round(permanencia.loc[permanencia['taxa'] < 0.8, 'silhueta'].mean(), 3))

lojas por taxa de permanência (100 subamostras de 80%):
              lojas   pct
taxa                     
até 50%          81   7.3
50-80%          194  17.4
80-95%          292  26.2
acima de 95%    548  49.1

por cluster:
         lojas  permanencia_media  pct_lojas_abaixo_80  silhueta_media
cluster                                                               
0          343              0.930               12.245           0.229
1          147              0.777               36.054           0.092
2          258              0.868               27.519           0.224
3          179              0.746               42.458           0.129
4          188              0.888               15.957           0.146

correlação de Spearman entre silhueta da loja e taxa de permanência: 0.585
silhueta média - lojas estáveis (>95%): 0.241 | instáveis (<80%): 0.085


In [6]:
permanencia.to_pickle(ESTADO / 'permanencia_lojas.pkl')

In [2]:
# o modelo de referência (semente 42) é a melhor solução que o K-Means encontra nesta base?
print(f'inércia do modelo de referência (semente 42, n_init=20): {modelo_k5.inertia_:.1f}')
modelo_exaustivo = KMeans(n_clusters=K, n_init=500, random_state=SEED).fit(X_modelo)
rot_exaustivo = alinhar(referencia.values, modelo_exaustivo.labels_)
print(f'inércia com n_init=500: {modelo_exaustivo.inertia_:.1f} ({(modelo_k5.inertia_ / modelo_exaustivo.inertia_ - 1) * 100:.2f}% menor que a referência)')
print('ARI referência x n_init=500:', round(adjusted_rand_score(referencia, modelo_exaustivo.labels_), 3),
      '| lojas em cluster diferente:', int((rot_exaustivo != referencia.values).sum()))
print('tamanhos referência :', np.bincount(referencia).tolist())
print('tamanhos n_init=500 :', np.bincount(rot_exaustivo).tolist())

# estabilidade medida contra a solução de menor inércia
ari_s, ari_sub = [], []
for i in range(30):
    ari_s.append(adjusted_rand_score(modelo_exaustivo.labels_, KMeans(n_clusters=K, n_init=20, random_state=1000 + i).fit_predict(X_modelo)))
    sub = X_modelo.sample(frac=0.8, random_state=3000 + i)
    ari_sub.append(adjusted_rand_score(pd.Series(modelo_exaustivo.labels_, index=X_modelo.index).loc[sub.index],
                                       KMeans(n_clusters=K, n_init=20, random_state=SEED).fit_predict(sub)))
print(f'\ncontra a solução de menor inércia: ARI semente médio {np.mean(ari_s):.3f} (mín {np.min(ari_s):.3f}) '
      f'| ARI subamostra80 médio {np.mean(ari_sub):.3f} (mín {np.min(ari_sub):.3f})')

inércia do modelo de referência (semente 42, n_init=20): 3890.5


inércia com n_init=500: 3884.0 (0.17% menor que a referência)
ARI referência x n_init=500: 0.908 | lojas em cluster diferente: 53
tamanhos referência : [343, 147, 258, 179, 188]
tamanhos n_init=500 : [345, 152, 269, 164, 185]



contra a solução de menor inércia: ARI semente médio 0.950 (mín 0.807) | ARI subamostra80 médio 0.792 (mín 0.537)


## 2. Os clusters são melhores que o cadastro?
### Teste A — concordância com tipo_loja e sortimento

In [3]:
concordancia = pd.DataFrame({
    'ARI': {c: adjusted_rand_score(base[c], referencia) for c in ['tipo_loja', 'sortimento']},
    'NMI': {c: normalized_mutual_info_score(base[c], referencia) for c in ['tipo_loja', 'sortimento']},
})
print(concordancia.round(3).to_string())
print('\ncluster x tipo_loja (% de cada tipo em cada cluster):')
print((pd.crosstab(base['tipo_loja'], referencia, normalize='index') * 100).round(1).to_string())
print('\ncluster x sortimento (% de cada sortimento em cada cluster):')
print((pd.crosstab(base['sortimento'], referencia, normalize='index') * 100).round(1).to_string())

              ARI    NMI
tipo_loja   0.152  0.174
sortimento  0.038  0.045

cluster x tipo_loja (% de cada tipo em cada cluster):
cluster       0     1     2     3     4
tipo_loja                              
a          10.3  14.0  33.1  18.8  23.9
b           0.0  23.5   5.9   0.0  70.6
c          23.0  33.8   9.5  20.9  12.8
d          71.0   2.6  12.6  10.1   3.7

cluster x sortimento (% de cada sortimento em cada cluster):
cluster        0     1     2     3     4
sortimento                              
a           20.2  10.6  28.0  20.9  20.2
b            0.0  44.4   0.0   0.0  55.6
c           43.5  15.6  17.9  10.7  12.3


### Teste B — variância explicada (eta²)
eta² = variação entre as médias dos grupos / variação total. É o R² de explicar a métrica só com o grupo da loja.

In [4]:
def eta2(valores, grupos):
    media_geral = valores.mean()
    entre = valores.groupby(grupos).agg(lambda s: len(s) * (s.mean() - media_geral) ** 2).sum()
    return entre / ((valores - media_geral) ** 2).sum()

metricas_negocio = {
    'venda_media_dia': 'no modelo (em log)', 'ticket_medio': 'no modelo', 'efeito_promo': 'no modelo',
    'sazonalidade': 'no modelo (em log)', 'pico_sabado': 'no modelo', 'distancia_concorrente': 'no modelo (em log)',
    'clientes_medios_dia': 'fora do modelo', 'volatilidade': 'fora do modelo', 'prop_domingos_abertos': 'fora do modelo',
}
gerador = np.random.default_rng(SEED)
linhas = []
for metrica, origem in metricas_negocio.items():
    aleatorio = np.mean([eta2(base[metrica], pd.Series(gerador.permutation(referencia.values), index=base.index)) for _ in range(200)])
    linhas.append({'metrica': metrica, 'origem': origem,
                   'tipo_loja': eta2(base[metrica], base['tipo_loja']) * 100,
                   'sortimento': eta2(base[metrica], base['sortimento']) * 100,
                   'cluster': eta2(base[metrica], referencia) * 100,
                   'cluster_aleatorio': aleatorio * 100})
tabela_eta = pd.DataFrame(linhas).set_index('metrica')
tabela_eta['ganho_pp'] = tabela_eta['cluster'] - tabela_eta['tipo_loja']
tabela_eta['vezes'] = tabela_eta['cluster'] / tabela_eta['tipo_loja']
print(tabela_eta.round(1).to_string())
print('\nmédia - no modelo  : tipo_loja', round(tabela_eta[tabela_eta['origem'].str.startswith('no')]['tipo_loja'].mean(), 1),
      '| cluster', round(tabela_eta[tabela_eta['origem'].str.startswith('no')]['cluster'].mean(), 1))
print('média - fora do modelo: tipo_loja', round(tabela_eta[tabela_eta['origem'] == 'fora do modelo']['tipo_loja'].mean(), 1),
      '| cluster', round(tabela_eta[tabela_eta['origem'] == 'fora do modelo']['cluster'].mean(), 1))

                                   origem  tipo_loja  sortimento  cluster  cluster_aleatorio  ganho_pp  vezes
metrica                                                                                                      
venda_media_dia        no modelo (em log)        2.8         2.5     14.5                0.4      11.7    5.2
ticket_medio                    no modelo       42.3        11.1     54.1                0.3      11.8    1.3
efeito_promo                    no modelo        6.0         6.9     39.2                0.3      33.2    6.5
sazonalidade           no modelo (em log)        2.9         5.0     41.3                0.4      38.3   14.0
pico_sabado                     no modelo       22.4         5.4     47.6                0.3      25.2    2.1
distancia_concorrente  no modelo (em log)        2.5         2.4     19.6                0.3      17.0    7.7
clientes_medios_dia        fora do modelo       25.5        10.9     25.3                0.4      -0.2    1.0
volatilida

In [5]:
# comparação com o mesmo número de grupos do tipo_loja (k = 4)
rot_k4 = pd.Series(KMeans(n_clusters=4, n_init=20, random_state=SEED).fit_predict(X_modelo), index=base.index)
print(pd.DataFrame({'tipo_loja (4 grupos)': {m: eta2(base[m], base['tipo_loja']) * 100 for m in metricas_negocio},
                    'cluster k=4': {m: eta2(base[m], rot_k4) * 100 for m in metricas_negocio}}).round(1).to_string())

                       tipo_loja (4 grupos)  cluster k=4
venda_media_dia                         2.8         12.9
ticket_medio                           42.3         55.5
efeito_promo                            6.0         18.5
sazonalidade                            2.9         25.7
pico_sabado                            22.4         48.1
distancia_concorrente                   2.5         17.6
clientes_medios_dia                    25.5         25.1
volatilidade                            2.0         39.2
prop_domingos_abertos                  70.0          1.8


## 3. Perfil médio de cada cluster

In [6]:
base['abre_domingo'] = (base['prop_domingos_abertos'] > 0).astype(int)
perfil = base.groupby('cluster').agg(
    lojas=('cluster', 'size'),
    venda_media_dia=('venda_media_dia', 'mean'),
    clientes_dia=('clientes_medios_dia', 'mean'),
    ticket_medio=('ticket_medio', 'mean'),
    efeito_promo=('efeito_promo', 'mean'),
    volatilidade=('volatilidade', 'mean'),
    sazonalidade=('sazonalidade', 'mean'),
    pico_sabado=('pico_sabado', 'mean'),
    distancia_mediana=('distancia_concorrente', 'median'),
    promo_continua=('promo_continua', 'mean'),
    prop_domingos_abertos=('prop_domingos_abertos', 'mean'),
    lojas_abrem_domingo=('abre_domingo', 'sum'),
)
rede = base.agg({'venda_media_dia': 'mean', 'clientes_medios_dia': 'mean', 'ticket_medio': 'mean', 'efeito_promo': 'mean',
                 'volatilidade': 'mean', 'sazonalidade': 'mean', 'pico_sabado': 'mean', 'distancia_concorrente': 'median',
                 'promo_continua': 'mean', 'prop_domingos_abertos': 'mean'})
perfil_t = perfil.T
perfil_t.columns = [f'cluster {c}' for c in perfil_t.columns]
perfil_t['rede'] = [len(base), rede['venda_media_dia'], rede['clientes_medios_dia'], rede['ticket_medio'], rede['efeito_promo'],
                    rede['volatilidade'], rede['sazonalidade'], rede['pico_sabado'], rede['distancia_concorrente'],
                    rede['promo_continua'], rede['prop_domingos_abertos'], base['abre_domingo'].sum()]
print(perfil_t.round(3).to_string())

                       cluster 0  cluster 1  cluster 2  cluster 3  cluster 4      rede
lojas                    343.000    147.000    258.000    179.000    188.000  1115.000
venda_media_dia         7136.159   7391.425   5848.257   6119.377   8476.883  6934.632
clientes_dia             616.359    967.888    631.030    701.125   1060.335   754.565
ticket_medio              11.721      7.855      9.311      8.985      8.337     9.644
efeito_promo               0.407      0.317      0.414      0.545      0.293     0.399
volatilidade               0.250      0.265      0.314      0.313      0.218     0.272
sazonalidade               0.093      0.131      0.099      0.095      0.069     0.096
pico_sabado                0.969      0.936      0.587      0.744      0.833     0.817
distancia_mediana       5470.000    700.000   5155.000    350.000   1135.000  2325.000
promo_continua             0.624      0.463      0.430      0.682      0.298     0.512
prop_domingos_abertos      0.006      0.064

In [7]:
perfil.to_pickle(ESTADO / 'perfil_clusters.pkl')
tabela_eta.to_pickle(ESTADO / 'tabela_eta.pkl')